## Libraries

In [2]:
import fast_text_splitter as fts
from langchain_text_splitters import RecursiveCharacterTextSplitter
import recursive_splitter_rust
import time
import random

## Splitter Definitions

In [3]:
# Cache basic setup
conf_params = fts.py_ws_config_params(None,["\n\n", "\n"],8, max_depth=10, parallel=False)
data = "Hello, you all! \n How are you foo and poo? \n\n I am fine. Nice to meet you all insecure!"

In [4]:
t = time.time()
for i in range(100):
    conf_params = fts.py_ws_config_params(None,["\n\n", "\n"],8, max_depth=10, parallel=False)
    splits = fts.text_split_ws(data, conf_params)
total = time.time() - t
time.sleep(1)
print("FastTextSplitter (cached):         ", round(total, 6))

FastTextSplitter (cached):          0.004819


In [5]:
t = time.time()
for i in range(100):
    # Randint is added to avoid cache hits (in ~97% of cases here), also costs only 200ns so absolutely negligible
    conf_params = fts.py_ws_config_params(None,["\n\n", "\n"],8, max_depth=10 + random.randint(1, 2 * 10**3), parallel=False) 
    splits = fts.text_split_ws(data, conf_params)
total = time.time() - t
time.sleep(1)
print("FastTextSplitter (non-cached): ", round(total,6))

FastTextSplitter (non-cached):  0.06797


In [6]:
if False:
    hf_conf_params = fts.py_hf_config_params(None,None,["\n\n", "\n"],32,2,True);
    splits = fts.text_split_hf(data, hf_conf_params)
    for split in splits:
        print("Split")
        print(split.split_strings)

In [7]:
#%%timeit
#random.randint(0, 10**6)

# OLD STUFF NOT UPDATED

In [8]:
splitter_rust = recursive_splitter_rust.RecursiveTextSplitter(
    target_length=300,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""],
    is_regex=False,
    trim_trailing = True
)

In [34]:
# fts
'''ArithmeticError#[pyo3(get)]
    pub pattern: Option<Vec<String>>,
    #[cfg(feature = "tokenizers")]
    #[pyo3(get)]
    pub model_path: Option<String>,
    #[pyo3(get)]
    pub max_tokens: Option<usize>,
    #[pyo3(get)]
    pub max_depth: Option<usize>,
    #[pyo3(get)]
    pub merge_level: Option<usize>,
    #[pyo3(get)]
    pub parallel: Option<bool>,
    #[pyo3(get)]
    pub conf_type: Option<String>,
'''
conf_params = fts.py_ws_config_params(patterns=["\n\n", "\n", ". "], max_tokens=55, max_depth=3, merge_level=1, parallel=False) 

In [35]:
splitter_langchain = RecursiveCharacterTextSplitter(
    chunk_size=300,
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_overlap=0,
    keep_separator=True,
    is_separator_regex=False,
    strip_whitespace=True,
)

## Text Load

In [36]:
# Open file
with open("../../superlinear.txt", "r") as file:
    contents = file.read()

## Quality Compare

In [37]:
# Quality compare 
results_lc = splitter_langchain.create_documents([contents])
results_fast = fts.text_split_ws(contents, conf_params)
results_rust = splitter_rust.split_texts([contents])[0]
print(len(results_lc), len(results_fast), len(results_rust), "\n")

for i, (text_lc, text_fast, text_rust) in enumerate(zip(results_lc, results_fast, results_rust)):
    print(f"Chunk {i + 1}:")
    print(f"Langchain: {text_lc.page_content}")
    print(f"FastText: {text_fast.split_strings}")
    print(f"RecursiveRust: {text_rust}")
    print("\n")
    if i == 2:
        break

123 111 116 

Chunk 1:
Langchain: October 2023

One of the most important things I didn't understand about the world when I was a child is the degree to which the returns for performance are superlinear.
FastText: October 2023

One of the most important things I didn't understand about the world when I was a child is the degree to which the returns for performance are superlinear.
RecursiveRust: October 2023

One of the most important things I didn't understand about the world when I was a child is the degree to which the returns for performance are superlinear.


Chunk 2:
Langchain: Teachers and coaches implicitly told us the returns were linear. "You get out," I heard a thousand times, "what you put in." They meant well, but this is rarely true. If your product is only half as good as your competitor's, you don't get half as many customers
FastText: 

Teachers and coaches implicitly told us the returns were linear. "You get out," I heard a thousand times, "what you put in." They mean

In [39]:
#for i, result in enumerate(results_fast):
#    print(f"Chunk {i + 1}: {result.split_strings}")

In [41]:
#for i, text in enumerate(results_rust):
#    print(f"Chunk {i + 1}: {text}")
#    print("-----------------------------")

## Speed Compare

In [42]:
copied = [contents]

In [46]:
%%timeit
results = splitter_langchain.create_documents(copied)

912 µs ± 3.86 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [47]:
%%timeit
results = splitter_rust.split_texts(copied)

47.6 µs ± 75.1 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [48]:
%%timeit
results = fts.text_split_ws(copied[0], conf_params)

1.17 ms ± 8.46 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
